In [2]:
# ============================================================
# NOTEBOOK 05 – GenAI Advice Generation
# Generate personalized financial advice using Groq LLM
# Requires GROQ_API_KEY set in .env
# ============================================================
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

print('✅ Project root:', PROJECT_ROOT)


✅ Project root: C:\Users\SomnathYadav\PycharmProjects\virtual-financial-advisor


In [3]:
# ----------------------------------------------------------
# Cell 1 – Load data
# ----------------------------------------------------------
import pandas as pd
from IPython.display import display

DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'virtual_financial_advisor_data.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f'Loaded {len(df):,} rows')

USER_ID = 'user_1'


Loaded 5,200 rows


In [4]:
# ----------------------------------------------------------
# Cell 2 – Gather real user context from analysis modules
# ----------------------------------------------------------
from src.trend_detection import financial_health_metrics, detect_risky_patterns
from src.scenario_simulation import (
    simulate_savings_increase,
    simulate_expense_reduction,
    simulate_investment_growth,
)

metrics = financial_health_metrics(df, USER_ID)
risks   = detect_risky_patterns(df, USER_ID)

print(f'Income        : ${metrics["total_income"]:,.2f}')
print(f'Expenses      : ${metrics["total_expenses"]:,.2f}')
print(f'Savings Rate  : {metrics["savings_rate"]*100:.1f}%')
print(f'Risks found   : {len(risks)}')


Income        : $165,306.82
Expenses      : $58,181.77
Savings Rate  : 65.0%
Risks found   : 0


In [1]:
# ----------------------------------------------------------
# Cell 3 – Basic advice: plain context string
# ----------------------------------------------------------
from src.genai_interface import generate_advice

basic_context = (
    "User saves less than 5% of income and has high dining expenses. "
    "Suggest 3 actionable steps to improve financial health."
)

print('=== Basic Advice ===')
advice = generate_advice(basic_context)
print(advice)


C:\Users\SomnathYadav\PycharmProjects\virtual-financial-advisor\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


=== Basic Advice ===
To improve financial health, consider the following steps:

1. **Track and reduce dining expenses**: Monitor dining spending for a month to identify areas for reduction. Aim to cut back by 20-30% by cooking at home more often, using coupons, or opting for affordable dining options.
2. **Implement a 10% savings rule**: Gradually increase savings rate to 10% of income by setting up automatic transfers from your checking account to your savings or investment accounts.
3. **Create a budget with a 50/30/20 allocation**: Allocate 50% of income towards necessary expenses (rent, utilities, groceries), 30% towards discretionary spending (dining, entertainment), and 20% towards saving and debt repayment. This framework will help prioritize needs over wants and build a safety net.


In [ ]:
# ----------------------------------------------------------
# Cell 4 – Data-backed advice built from real metrics
# ----------------------------------------------------------
top_categories = list(metrics['category_spend'].keys())[:3]
risk_summary   = '; '.join(r['message'] for r in risks) if risks else 'No major risks.'

rich_context = (
    f"User ID       : {USER_ID}\n"
    f"Total Income  : ${metrics['total_income']:,.2f}\n"
    f"Total Expenses: ${metrics['total_expenses']:,.2f}\n"
    f"Savings Rate  : {metrics['savings_rate']*100:.1f}%\n"
    f"Top Categories: {top_categories}\n"
    f"Risk Summary  : {risk_summary}\n\n"
    "Provide exactly 3 specific, data-backed recommendations to improve "
    "financial health, ordered by potential impact."
)

print('=== Data-Backed Advice ===')
rich_advice = generate_advice(rich_context)
print(rich_advice)


In [ ]:
# ----------------------------------------------------------
# Cell 5 – Scenario-aware advice
# ----------------------------------------------------------
s1 = simulate_savings_increase(df, USER_ID, 0.10, 12)
s2 = simulate_expense_reduction(df, USER_ID, 0.15, 12)
s3 = simulate_investment_growth(df, USER_ID, 0.07, 10)

scenario_context = (
    f"Current savings rate: {metrics['savings_rate']*100:.1f}%.\n"
    f"What-if scenarios:\n"
    f"  • Saving 10% more for 12 months yields an extra ${s1['extra_savings']:,.2f}.\n"
    f"  • Cutting expenses by 15% for 12 months saves ${s2['total_savings_gained']:,.2f}.\n"
    f"  • Investing current savings at 7% for 10 years grows to "
    f"${s3['projected_portfolio_value']:,.2f}.\n\n"
    "Which scenario gives the best long-term outcome? "
    "Give a brief recommendation with concrete numbers."
)

print('=== Scenario-Aware Advice ===')
scenario_advice = generate_advice(scenario_context)
print(scenario_advice)


In [ ]:
# ----------------------------------------------------------
# Cell 6 – Custom question (edit and re-run)
# ----------------------------------------------------------
custom_question = (
    "I want to buy a house in 5 years. Based on my current savings rate of "
    f"{metrics['savings_rate']*100:.1f}%, what should I do to reach a 30% "
    "savings rate and build a down-payment fund?"
)

print('=== Custom Question Advice ===')
print('Q:', custom_question)
print()
custom_advice = generate_advice(custom_question)
print(custom_advice)


In [ ]:
# ----------------------------------------------------------
# Cell 7 – Advice for all users (batch)
# ----------------------------------------------------------
all_users = sorted(df['user_id'].unique())
print('=== Batch Advice – All Users ===\n')
for uid in all_users:
    m = financial_health_metrics(df, uid)
    ctx = (
        f"User {uid}: income ${m['total_income']:,.0f}, "
        f"expenses ${m['total_expenses']:,.0f}, "
        f"savings rate {m['savings_rate']*100:.1f}%. "
        "Give one key recommendation in one sentence."
    )
    try:
        tip = generate_advice(ctx)
        print(f"[{uid}] {tip}\n")
    except Exception as e:
        print(f"[{uid}] Error: {e}\n")
